In [4]:
import pandas as pd
import json
import ast
from concurrent.futures import ThreadPoolExecutor, as_completed


# Import your existing conversation generator
import sys
sys.path.append('/home/sagemaker-user/csbai/multiturn_rl')
from simulators.conversation_simulator import ConversationConfig, MultiTurnConversationGenerator

from utils.bedrock_call import bedrock_call

# Interactivity evaluation prompt
INTERACTIVITY_PROMPT = '''You are a helpful and meticulous conversation evaluator. \
Your task is to evaluate the *interactivity* of the responses provided by an AI assistant \
to user questions in a given conversation:
<|The Start of the Conversation to be Evaluated|>
{chat_history}
<|The End of the Conversation to be Evaluated|>
You should assess the assistant's engagement, clarity, and ability to understand the user's needs. Evaluate the style of the conversation not the length. Longer conversation does not mean more interactive. \
Give a float number between 0 and 1, where:
    1 = Highly interactive: The assistant is very engaging, asks all relevant questions, and significantly enhances understanding of user preference.
     - Example: The assistant thoroughly understands the user's question, asks for necessary clarifications.
    0.5 = Moderately interactive: The assistant is engaging, asks some relevant questions, but can be substantially improved.
     - Example: The assistant asks some relevant questions about the user's inquiry but misses key details, and does not probe further for clarification.
    0 = Low interactivity: The assistant shows low engagement, asks few relevant questions, and barely try to understand the user's needs.
     - Example: The assistant provides a vague or incomplete response without fully understanding the user's intent, such as "You should watch this movie" without asking any follow-up questions or providing detailed information.
Output format (JSON):
{{
    "thought": "<How interactive is the assistant?>",
    "interactivity": <score>
}}
Double check if the JSON object is formatted correctly. Ensure that all fields are present and properly structured. Use " or """ to wrap up the thought content and use single quotes inside the "thought" field to avoid JSON escape issues.
Your evaluation:
'''

def parse_conversation(conversation_str):
    """Parse conversation string to extract chat history"""
    try:
        # Handle the conversation format from your data
        conversation_list = ast.literal_eval(conversation_str)
        if len(conversation_list) > 10:
            conversation_list = conversation_list[6:16]
        
        # Format conversation for evaluation
        chat_history = ""
        for turn in conversation_list:
            role = turn.get('role', '')
            content = turn.get('content', '')
            # Clean up quotation marks
            content = content.replace('QUOTATION_MARK', '"')
            chat_history += f"{role.capitalize()}: {content}\n\n"
        
        return chat_history.strip()
    except Exception as e:
        print(f"Error parsing conversation: {e}")
        return ""

def extract_score_from_response(response):
    """Extract interactivity score from LLM response"""
    try:
        # Try to find JSON in the response
        start_idx = response.find('{')
        end_idx = response.rfind('}') + 1
        
        if start_idx != -1 and end_idx != -1:
            json_str = response[start_idx:end_idx]
            result = json.loads(json_str)
            return float(result.get('interactivity', 0))
        else:
            # Fallback: try to extract number directly
            import re
            numbers = re.findall(r'"interactivity":\s*([0-9.]+)', response)
            if numbers:
                return float(numbers[0])
            
        print(f"Could not extract score from response: {response[:200]}...")
        return None
    except Exception as e:
        print(f"Error extracting score: {e}")
        return None

def evaluate_single_conversation(row_data):
    """Evaluate a single conversation for interactivity"""
    try:
        index, generated_conversation = row_data
        
        # Parse the conversation
        chat_history = parse_conversation(generated_conversation)
        if not chat_history:
            return index, None
        
        # Create the evaluation prompt
        prompt = INTERACTIVITY_PROMPT.format(chat_history=chat_history)
        
        # Prepare messages for the model
        messages = [
            {"role": "user", "content": prompt}
        ]
        
        # Call the model (using Claude-3 Haiku for efficiency)
        model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
        response = bedrock_call(
            model=model_id,
            messages=messages,
            max_tokens=1000,
            temperature=0.1
        )
        
        if response is None:
            return index, None
        
        # Extract score from response
        score = extract_score_from_response(response)
        return index, score
        
    except Exception as e:
        print(f"Error evaluating conversation at index {index}: {e}")
        return index, None

def evaluate_conversations_parallel(csv_file_path, output_file_path, max_workers=10):
    """
    Evaluate conversations in parallel and add interactivity scores to CSV
    
    Args:
        csv_file_path: Path to input CSV file
        output_file_path: Path to output CSV file
        max_workers: Number of parallel workers
    """
    
    # Read the CSV file
    print("Reading CSV file...")
    df = pd.read_csv(csv_file_path)
    
    print(f"Found {len(df)} conversations to evaluate")
    
    # Prepare data for parallel processing
    conversations_data = [(idx, row['generated_conversation']) 
                         for idx, row in df.iterrows()]
    
    # Initialize results dictionary
    results = {}
    
    # Process conversations in parallel
    print(f"Starting parallel evaluation with {max_workers} workers...")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_index = {
            executor.submit(evaluate_single_conversation, conv_data): conv_data[0] 
            for conv_data in conversations_data
        }
        
        # Collect results as they complete
        completed = 0
        for future in as_completed(future_to_index):
            index, score = future.result()
            results[index] = score
            completed += 1
            
            if completed % 10 == 0:
                print(f"Completed {completed}/{len(conversations_data)} evaluations")
    
    # Add scores to dataframe
    print("Adding scores to dataframe...")
    df['interactivity_score'] = df.index.map(results)
    
    # Save results
    df.to_csv(output_file_path, index=False)
    print(f"Results saved to {output_file_path}")
    
    # Print summary statistics
    valid_scores = df['interactivity_score'].dropna()
    if len(valid_scores) > 0:
        print(f"\nSummary Statistics:")
        print(f"Successfully evaluated: {len(valid_scores)}/{len(df)} conversations")
        print(f"Average interactivity score: {valid_scores.mean():.3f}")
        print(f"Score range: {valid_scores.min():.3f} - {valid_scores.max():.3f}")
        print(f"Standard deviation: {valid_scores.std():.3f}")
    else:
        print("No valid scores obtained!")

# Example usage
if __name__ == "__main__":
    # Set your file paths
    dataset = "redial"
    alg = "vanilla"
    model = 'llama-3.2-instruct'
    folder_path = f"../generated_testsets/multiturn_test/{alg}/{model}/{dataset}/"
    input_csv_path = folder_path + "generated_movie_conversations.csv"  # Your input CSV file
    # input_csv_path = "../datasets/redial/multiturn_form/train.csv"
    output_csv_path = folder_path + "interactivity_score.csv"  # Output file with scores
    
    # Run the evaluation
    evaluate_conversations_parallel(
        csv_file_path=input_csv_path,
        output_file_path=output_csv_path,
        max_workers=50  # Adjust based on your AWS limits
    )

Reading CSV file...
Found 1076 conversations to evaluate
Starting parallel evaluation with 50 workers...
Completed 10/1076 evaluations
Completed 20/1076 evaluations
Completed 30/1076 evaluations
Completed 40/1076 evaluations
Completed 50/1076 evaluations
Completed 60/1076 evaluations
Completed 70/1076 evaluations
Completed 80/1076 evaluations
Completed 90/1076 evaluations
Completed 100/1076 evaluations
Completed 110/1076 evaluations
Completed 120/1076 evaluations
Completed 130/1076 evaluations
Completed 140/1076 evaluations
Completed 150/1076 evaluations
Completed 160/1076 evaluations
Completed 170/1076 evaluations
Completed 180/1076 evaluations
Completed 190/1076 evaluations
Completed 200/1076 evaluations
Completed 210/1076 evaluations
Completed 220/1076 evaluations
Completed 230/1076 evaluations
Completed 240/1076 evaluations
Completed 250/1076 evaluations
Completed 260/1076 evaluations
Completed 270/1076 evaluations
Completed 280/1076 evaluations
Completed 290/1076 evaluations
Compl